In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

import tensorflow as tf
import tensorflow_datasets as tfds

print("TF Version: ", tf.__version__)
print("TF Eager mode: ", tf.executing_eagerly())
print("TF GPU is", "available" if tf.config.list_physical_devices("GPU") else "not available")

In [ ]:
# The size of a batch
BATCH_SIZE = 32
# The image size to use MobileNetV2 pre-trained model
IMAGE_SIZE = (224, 224)
# The image shape
IMAGE_SHAPE = IMAGE_SIZE + (3,)

## Prepare Datasets

In [ ]:
# Download dataset directly from URL
data_dir = tf.keras.utils.get_file(
    origin="https://storage.googleapis.com/"
           "tensorflow-3-public/datasets/caltech_birds2010_011.zip",
    extract=True)

In [ ]:
(raw_tr_ds, raw_ts_ds), ds_info = tfds.load("caltech_birds2010",
                                            split=["train", "test"],
                                            with_info=True,
                                            data_dir=data_dir,
                                            download=False)

In [ ]:
print(f"Number of training examples: {len(raw_tr_ds)}")
print(f"Number of test examples: {len(raw_ts_ds)}")

In [ ]:
def preprocess_sample(data):
    # Get image and bbox
    in_image, in_bbox = data["image"], data["bbox"]
    # Update bbox considering new image size
    s_shape = tf.cast(tf.shape(in_image), dtype=tf.float32)
    (sx, sy) = s_shape[1], s_shape[0]
    d_shape = tf.cast(IMAGE_SHAPE, dtype=tf.float32)
    (dx, dy) = d_shape[1], d_shape[0]
    out_bbox = [(in_bbox[1] * sx) / dx,
                (in_bbox[0] * sy) / dy,
                (in_bbox[3] * sx) / dx,
                (in_bbox[2] * sy) / dy]
    # Resize image to fit model
    out_image = tf.cast(in_image, dtype=tf.float32)
    out_image = tf.image.resize(out_image, size=IMAGE_SIZE)
    # Pre-process image to fit model
    out_image = tf.keras.applications.mobilenet.preprocess_input(out_image)
    return out_image, out_bbox

In [ ]:
tr_ds = (raw_tr_ds
         .map(preprocess_sample)
         .shuffle(buffer_size=1000)
         .batch(BATCH_SIZE)
         .prefetch(tf.data.AUTOTUNE))

ts_ds = (raw_ts_ds
         .map(preprocess_sample)
         .cache()
         .batch(BATCH_SIZE)
         .prefetch(tf.data.AUTOTUNE))

## Create Model

* Get the architecture of existing model
* Use existing weights
* Add layers to predict bounding boxes (regression)

In [ ]:
inputs = tf.keras.layers.Input(shape=IMAGE_SHAPE)
x = tf.keras.applications.MobileNetV2(
      input_shape=IMAGE_SHAPE,
      include_top=False,
      weights="imagenet"
)(inputs)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(1024, activation="relu")(x)
x = tf.keras.layers.Dense(512, activation="relu")(x)
outputs = tf.keras.layers.Dense(4)(x) # 4 units as we need to predict bbox (xmin, ymin, xmax, ymax)
model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.SGD(momentum=0.9),
    loss="mse")

### Train Model

In [ ]:
EPOCHS = 3

tr_len = len(raw_tr_ds)
ts_len = len(raw_ts_ds)
tr_steps = math.ceil(tr_len / BATCH_SIZE)
ts_steps = math.ceil(ts_len / BATCH_SIZE)

history = model.fit(
    tr_ds,
    steps_per_epoch=tr_steps,
    validation_data=ts_ds,
    validation_steps=ts_steps,
    epochs=EPOCHS)

### Evaluate Model

In [ ]:
loss = model.evaluate(ts_ds, steps=ts_steps)

#### Plot Metrics

In [ ]:
def plot_metrics(metric_name, title, ylim=5):
    plt.title(title)
    plt.ylim(0,ylim)
    plt.plot(history.history[metric_name],color='blue',label=metric_name)
    plt.plot(history.history['val_' + metric_name],color='green',label='val_' + metric_name)

In [ ]:
plot_metrics("loss", "Bounding Box Loss", ylim=0.2)

#### Calculate IoU

In [ ]:
def intersection_over_union(pred_box, true_box):
    xmin_pred, ymin_pred, xmax_pred, ymax_pred =  np.split(pred_box, 4, axis = 1)
    xmin_true, ymin_true, xmax_true, ymax_true = np.split(true_box, 4, axis = 1)
    # Calculate coordinates of overlap area between boxes
    xmin_overlap = np.maximum(xmin_pred, xmin_true)
    xmax_overlap = np.minimum(xmax_pred, xmax_true)
    ymin_overlap = np.maximum(ymin_pred, ymin_true)
    ymax_overlap = np.minimum(ymax_pred, ymax_true)
    # Calculates area of true and predicted boxes
    pred_box_area = (xmax_pred - xmin_pred) * (ymax_pred - ymin_pred)
    true_box_area = (xmax_true - xmin_true) * (ymax_true - ymin_true)
    # Calculates overlap area and union area.
    overlap_area = np.maximum((xmax_overlap - xmin_overlap),0)  * np.maximum((ymax_overlap - ymin_overlap), 0)
    union_area = (pred_box_area + true_box_area) - overlap_area
    # Defines a smoothing factor to prevent division by 0
    smoothing_factor = 1e-10
    # Updates iou score
    iou = (overlap_area + smoothing_factor) / (union_area + smoothing_factor)
    return iou

In [ ]:
# Split and convert to numpy array
it = list(ts_ds.unbatch().take(100).as_numpy_iterator())
true_images = np.array([item[0] for item in it])
true_bboxes = np.array([item[1] for item in it])

In [ ]:
# Predict bboxes
pred_bboxes = model.predict(true_images)

In [ ]:
# Calculate IoU
iou = intersection_over_union(pred_bboxes, true_bboxes)
iou_threshold = 0.5

In [ ]:
print(f"Number of predictions where iou >= threshold ({iou_threshold}): {(iou >= iou_threshold).sum()}")
print(f"Number of predictions where iou <  threshold ({iou_threshold}): {(iou <  iou_threshold).sum()}")